# 03 - Structural and Similarity Curve Terms

This notebook isolates the shape-aware curve losses: cosine dissimilarity,
spectral coherence, and SSIM. Unlike the pointwise terms in notebook 02,
these respond to the structure of the profile rather than raw amplitude
differences. We perturb a reference profile (vertical shift of the centre,
amplitude scaling, additive noise) and confirm each term increases with
increasing distortion.

Modules exercised: `LossComponents.cosine`, `LossComponents.spectral_coherence`,
`LossComponents.ssim`, `Loss.reconstruct_gaussians`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
from configuration.training_config import GaussianConfig, GeometryConfig, LossConfig
from pipelines.training_pipeline.loss import Loss, LossComponents as LC
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX, N_SAMPLES = -10.0, 40.0, 64
x_axis       = torch.linspace(X_MIN, X_MAX, N_SAMPLES, dtype=torch.float32)
gaussian_cfg = GaussianConfig(n_default_gaussians=1, x_min=X_MIN, x_max=X_MAX)

criterion = Loss(x_axis, NullLogger(), NullTracker(), gaussian_cfg,
                 LossConfig(use_ssim_curve=True), IdentityNormStats(), GeometryConfig())

def grid_curve(amp, mu, sig, H=4, W=4):
    p = torch.tensor([amp, mu, sig], dtype=torch.float32).reshape(1, 3, 1, 1)
    p = p.expand(1, 3, H, W).contiguous()
    return criterion.reconstruct_gaussians(p)

target = grid_curve(2.0, 10.0, 4.0)
print('target curve tensor shape (B, N, H, W):', tuple(target.shape))


## Response to a centre shift

We slide the predicted peak away from the target centre and evaluate cosine
dissimilarity, spectral coherence loss, and SSIM loss. All should rise from
zero (perfect overlap) as the shift grows.

In [ ]:
shifts = torch.linspace(0.0, 20.0, 41)
cos_v, coh_v, ssim_v = [], [], []

for s in shifts:
    pred = grid_curve(2.0, 10.0 + float(s), 4.0)
    cos_v.append( float(LC.cosine(pred, target, axis=1)) )
    coh_v.append( float(LC.spectral_coherence(pred, target, window=7)) )
    ssim_v.append(float(LC.ssim(pred, target, criterion.loss_cfg)) )

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(shifts, cos_v,  label='cosine dissimilarity')
ax.plot(shifts, coh_v,  label='spectral coherence loss')
ax.plot(shifts, ssim_v, label='SSIM loss')
ax.set_xlabel('centre shift [m]')
ax.set_ylabel('loss value')
ax.set_title('Structural terms vs peak displacement')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Amplitude scaling vs additive noise

Cosine similarity is scale-invariant by construction (it normalises each
profile), so pure amplitude scaling should leave it near zero while SSIM and
coherence may react. Additive noise degrades all structural terms.

In [ ]:
scales = torch.linspace(0.2, 3.0, 29)
cos_scale  = [float(LC.cosine(grid_curve(2.0 * float(k), 10.0, 4.0), target, axis=1)) for k in scales]
ssim_scale = [float(LC.ssim(grid_curve(2.0 * float(k), 10.0, 4.0), target, criterion.loss_cfg)) for k in scales]

noise_std  = torch.linspace(0.0, 1.0, 21)
torch.manual_seed(SEED)
cos_noise, ssim_noise = [], []
for sd in noise_std:
    pred = target + float(sd) * torch.randn_like(target)
    cos_noise.append( float(LC.cosine(pred, target, axis=1)) )
    ssim_noise.append(float(LC.ssim(pred, target, criterion.loss_cfg)) )

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(scales, cos_scale,  label='cosine')
axes[0].plot(scales, ssim_scale, label='SSIM')
axes[0].axvline(1.0, color='grey', ls=':', lw=1)
axes[0].set_xlabel('amplitude scale factor'); axes[0].set_title('amplitude scaling')
axes[0].legend(frameon=False)

axes[1].plot(noise_std, cos_noise,  label='cosine')
axes[1].plot(noise_std, ssim_noise, label='SSIM')
axes[1].set_xlabel('additive noise std'); axes[1].set_title('additive noise')
axes[1].legend(frameon=False)

for ax in axes:
    ax.set_ylabel('loss value')
fig.tight_layout()
plt.show()


## Expected visual outcome

All three structural terms should be near zero at zero displacement and grow
monotonically with peak shift. Under pure amplitude scaling the cosine term
stays close to zero (scale invariance) with a minimum at scale 1, whereas
SSIM reacts to contrast change. Additive noise raises both terms steadily.
This confirms the structural losses penalise shape mismatch as intended.